In [1]:
import pandas as pd

In [2]:
train = pd.read_csv('/Users/keltonn/Desktop/Projects/housing_price_regressor/data/scaled_static_features/train_feature.csv')
train

,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LEASE_COMMENCE_DATE,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,MONTHLY_RENT,...,mrt_traffic_weekday_tap_norm,mrt_traffic_weekend_tap_norm,mrt_traffic_am_peak_norm,mrt_traffic_pm_peak_norm,mrt_traffic_busiest_hour,nearest_mall,train_lines,station_is_open,n_malls_500m,nearest_mall_m
0,2021,1,2012,punggol,604a,punggol road,4 room,model a,-0.016241,1800,...,1.045435,1.353766,0.685497,1.022432,18,Waterway Point,3,1,0.477343,-0.828922
1,2021,1,2001,sengkang,290c,compassvale crescent,5 room,improved,0.684225,2000,...,1.242553,1.199175,1.102846,1.240234,18,Compass One,2,1,0.477343,-0.963672
2,2021,1,1976,queenstown,5,ghim moh road,3 room,improved,-1.169949,2000,...,0.358550,-0.113052,0.484020,0.635972,18,The Star Vista,2,1,0.477343,-0.776008
3,2021,1,2010,sengkang,436b,fernvale road,4 room,model a,0.107371,1400,...,1.242553,1.199175,1.102846,1.240234,18,Compass One,2,1,0.477343,-0.963672
4,2021,1,2001,sembawang,467b,admiralty drive,4 room,premium apartment,0.354594,1800,...,0.916294,0.893724,1.089398,0.839276,7,Sun Plaza,1,1,0.477343,-0.884465
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149354,2025,3,1981,kallang/whampoa,43,bendemeer road,3 room,standard,-1.252357,2900,...,-0.011608,-0.058126,0.039290,-0.024803,18,City Square Mall,1,1,-0.826239,0.654315
149355,2025,3,1985,bukit batok,220,bukit batok east avenue 3,3 room,new generation,-1.087541,2500,...,0.323756,0.253095,0.324322,0.392571,18,West Mall,1,1,0.477343,-0.895804
149356,2025,3,1971,toa payoh,17,lorong 7 toa payoh,3 room,improved,-1.087541,2650,...,0.686469,0.686486,0.628972,0.627443,18,HDB Hub,1,1,0.477343,-0.850449
149357,2025,3,2001,sembawang,360a,admiralty drive,4 room,premium apartment,0.024963,3300,...,0.916294,0.893724,1.089398,0.839276,7,Sun Plaza,1,1,0.477343,-0.884465


In [3]:
test = pd.read_csv('/Users/keltonn/Desktop/Projects/housing_price_regressor/data/original_dataset/test.csv')
test['RENT_APPROVAL_DATE'].min()

'2025-03'

In [4]:
df = pd.read_csv('/Users/keltonn/Desktop/Projects/housing_price_regressor/data/original_dataset/auxiliary/sg-stock-prices.csv')
df

,NAME,SYMBOL,DATE,OPEN,HIGH,LOW,CLOSE,ADJUSTED_CLOSE,VOLUME
0,DBS Group,D05.SI,2021-01-04,22.845,23.036,22.736,23.036,17.210,2372275.0
1,DBS Group,D05.SI,2021-01-05,22.927,23.045,22.809,23.045,17.217,2284480.0
2,DBS Group,D05.SI,2021-01-06,23.000,23.045,22.791,22.891,17.101,4188360.0
3,DBS Group,D05.SI,2021-01-07,23.345,23.800,23.300,23.682,17.692,13132460.0
4,DBS Group,D05.SI,2021-01-08,24.136,24.973,24.091,24.927,18.622,15961880.0
...,...,...,...,...,...,...,...,...,...
91637,EUDA Health,EUDA,2026-07-24,14.500,15.100,14.500,14.990,14.990,9900.0
91638,EUDA Health,EUDA,2026-07-27,14.920,19.270,14.920,19.000,19.000,16200.0
91639,EUDA Health,EUDA,2026-07-28,19.800,21.000,19.520,19.900,19.900,7200.0
91640,EUDA Health,EUDA,2026-07-29,19.970,20.953,19.364,20.350,20.350,5600.0


In [5]:
import numpy as np

KEY = ['RENT_APPROVAL_YEAR', 'RENT_APPROVAL_MONTH']

# 85 rows (non-trading days) have no prices at all
stock = df.dropna(subset=['ADJUSTED_CLOSE']).copy()
stock['DATE'] = pd.to_datetime(stock['DATE'])
stock['RENT_APPROVAL_YEAR'] = stock['DATE'].dt.year
stock['RENT_APPROVAL_MONTH'] = stock['DATE'].dt.month
stock = stock.sort_values(['NAME', 'DATE'])

# Everything is computed on the log of the adjusted close (accounts for dividends/splits), so daily changes are log
# returns and moves are comparable between cheap and expensive stocks.
stock['LOG_PRICE'] = np.log(stock['ADJUSTED_CLOSE'])
stock['DAILY_LOG_RETURN'] = stock.groupby('NAME')['LOG_PRICE'].diff()

# One row per stock x year+month: the month's average level and its day-to-day volatility
monthly = (
    stock.groupby(['NAME'] + KEY)
    .agg(
        LOG_PRICE_MEAN=('LOG_PRICE', 'mean'),
        RETURN_VOLATILITY=('DAILY_LOG_RETURN', 'std'),
    )
    .reset_index()
)
monthly.head()

,NAME,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LOG_PRICE_MEAN,RETURN_VOLATILITY
0,Antalpha Platform,2025,5,2.519028,0.135667
1,Antalpha Platform,2025,6,2.486008,0.056202
2,Antalpha Platform,2025,7,2.461962,0.039747
3,Antalpha Platform,2025,8,2.512603,0.031019
4,Antalpha Platform,2025,9,2.499707,0.025648


In [6]:
# Month-over-month return = change in the monthly *average* log price versus the previous month, so it reflects the
# month's overall conditions rather than where the last trading day happened to close. Done on a complete month
# index so a missing month can never be silently treated as the "previous" month. The first month of each stock is NaN.
monthly['MONTH'] = pd.PeriodIndex(year=monthly['RENT_APPROVAL_YEAR'], month=monthly['RENT_APPROVAL_MONTH'], freq='M')
mean_price = monthly.pivot(index='MONTH', columns='NAME', values='LOG_PRICE_MEAN')
mean_price = mean_price.reindex(pd.period_range(mean_price.index.min(), mean_price.index.max(), freq='M'))
mom = (
    mean_price.diff()
    .stack(dropna=False)
    .rename('MOM_RETURN')
    .rename_axis(['MONTH', 'NAME'])
    .reset_index()
)
monthly = monthly.merge(mom, on=['MONTH', 'NAME'], how='left').drop(columns=['MONTH'])
monthly.head()

,NAME,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,LOG_PRICE_MEAN,RETURN_VOLATILITY,MOM_RETURN
0,Antalpha Platform,2025,5,2.519028,0.135667,NaN
1,Antalpha Platform,2025,6,2.486008,0.056202,-0.033020
2,Antalpha Platform,2025,7,2.461962,0.039747,-0.024046
3,Antalpha Platform,2025,8,2.512603,0.031019,0.050641
4,Antalpha Platform,2025,9,2.499707,0.025648,-0.012897


In [7]:
# Pivot to one row per year+month (the join grain), one column per stock x feature.
# Stocks that were not yet listed in a given month are left as NaN (not 0) so the model can tell them apart.
features = monthly.melt(id_vars=['NAME'] + KEY, var_name='FEATURE')
features['COL'] = (
    'stk_'
    + features['NAME'].str.lower().str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_')
    + '_'
    + features['FEATURE'].str.lower()
)
stock_features = features.pivot(index=KEY, columns='COL', values='value').reset_index()
stock_features.columns.name = None
stock_features.shape

(67, 233)

In [8]:
# Check the join against the training set: every training row should find its month, and only the
# first stock month (2021-01, no prior month for MOM_RETURN) should be missing a return.
check = train[KEY].merge(stock_features, on=KEY, how='left', validate='many_to_one')
assert len(check) == len(train)
print('train rows without stock features:', check['stk_dbs_group_log_price_mean'].isna().sum())

stock_features.to_csv(
    '/Users/keltonn/Desktop/Projects/housing_price_regressor/data/temporal_features/stock_features.csv',
    index=False,
)
stock_features.head()

train rows without stock features: 0


,RENT_APPROVAL_YEAR,RENT_APPROVAL_MONTH,stk_antalpha_platform_log_price_mean,stk_antalpha_platform_mom_return,stk_antalpha_platform_return_volatility,stk_ascendas_reit_log_price_mean,stk_ascendas_reit_mom_return,stk_ascendas_reit_return_volatility,stk_belive_holdings_log_price_mean,stk_belive_holdings_mom_return,...,stk_wave_life_sciences_return_volatility,stk_webuy_global_log_price_mean,stk_webuy_global_mom_return,stk_webuy_global_return_volatility,stk_wilmar_international_log_price_mean,stk_wilmar_international_mom_return,stk_wilmar_international_return_volatility,stk_yanlord_log_price_mean,stk_yanlord_mom_return,stk_yanlord_return_volatility
0,2021,1,NaN,NaN,NaN,0.806770,NaN,0.012960,NaN,NaN,...,0.043841,NaN,NaN,NaN,1.402303,NaN,0.014903,0.001021,NaN,0.012827
1,2021,2,NaN,NaN,NaN,0.810418,0.003648,0.010391,NaN,NaN,...,0.043720,NaN,NaN,NaN,1.440095,0.037792,0.012453,-0.009466,-0.010488,0.007062
2,2021,3,NaN,NaN,NaN,0.786484,-0.023934,0.012611,NaN,NaN,...,0.075097,NaN,NaN,NaN,1.427988,-0.012107,0.008999,0.039096,0.048563,0.011887
3,2021,4,NaN,NaN,NaN,0.824610,0.038125,0.006253,NaN,NaN,...,0.027922,NaN,NaN,NaN,1.446436,0.018448,0.007545,0.133329,0.094233,0.011101
4,2021,5,NaN,NaN,NaN,0.792135,-0.032474,0.012063,NaN,NaN,...,0.046543,NaN,NaN,NaN,1.363611,-0.082825,0.015108,0.180473,0.047144,0.019049
